In [ ]:
# ================================
# Environment Fix for Kaggle (NumPy/SciPy/sklearn)
# ================================
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *pkgs])

pip_install([
    "numpy==1.26.4",       # last stable 1.x
    "scipy==1.11.4",       # compatible with NumPy 1.26
    "scikit-learn==1.3.2"  # compatible with above
])

print("✅ Installed fixed versions.")
print("⚠️ IMPORTANT: Go to 'Runtime' → 'Restart' now, then re-run from Cell 1.")



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install gensim==4.3.2 tqdm nltk

import os, math, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
warnings.filterwarnings("ignore")

# BLEU
import nltk
nltk.download("punkt")
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Setup complete | Device:", DEVICE)


In [ ]:
from gensim.models import Word2Vec

# Load pretrained Word2Vec model (choose: code, doc, combined)
w2v_doc = Word2Vec.load("/kaggle/input/word-to-vec/w2v_doc.model")

class W2VTokenizer:
    def __init__(self, w2v_model, add_specials=True):
        self.pad_token, self.unk_token, self.bos_token, self.eos_token = "<PAD>", "<UNK>", "<BOS>", "<EOS>"
        self.specials = [self.pad_token, self.unk_token, self.bos_token, self.eos_token] if add_specials else []

        self.token_to_id = {t:i for i,t in enumerate(self.specials)}
        offset = len(self.token_to_id)

        for i, w in enumerate(w2v_model.wv.index_to_key):
            self.token_to_id[w] = i + offset
        self.id_to_token = {i:t for t,i in self.token_to_id.items()}

        dim = w2v_model.vector_size
        self.embed_matrix = np.random.normal(scale=0.01, size=(len(self.token_to_id), dim)).astype(np.float32)
        for w, idx in self.token_to_id.items():
            if w in w2v_model.wv:
                self.embed_matrix[idx] = w2v_model.wv[w]

        self.pad_id = self.token_to_id[self.pad_token]
        self.unk_id = self.token_to_id[self.unk_token]
        self.bos_id = self.token_to_id[self.bos_token]
        self.eos_id = self.token_to_id[self.eos_token]

    def tokenize(self, text): return text.strip().split() if isinstance(text, str) else []
    def encode(self, text, max_len=256):
        toks = [self.bos_token] + self.tokenize(text) + [self.eos_token]
        return [self.token_to_id.get(t, self.unk_id) for t in toks][:max_len]
    def decode(self, ids):
        toks = [self.id_to_token.get(i, self.unk_id) for i in ids]
        return " ".join([t for t in toks if t not in self.specials])

tokenizer = W2VTokenizer(w2v_doc)
print("✅ Tokenizer ready | vocab size:", len(tokenizer.token_to_id))


In [ ]:
class LMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=256):
        self.examples = []
        for t in texts:
            ids = tokenizer.encode(t, max_len=max_len)
            self.examples.append(torch.tensor(ids, dtype=torch.long))
        self.pad_id = tokenizer.pad_id
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

def collate_batch(batch, pad_id):
    batch = pad_sequence(batch, batch_first=True, padding_value=pad_id)
    return batch[:, :-1], batch[:, 1:], torch.tensor([len(x)-1 for x in batch], dtype=torch.long)

# Load dataset
df = pd.read_csv("/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv")
texts = df["docstring"].fillna("").astype(str).tolist()
train_texts, tmp = train_test_split(texts, test_size=0.2, random_state=42)
val_texts, test_texts = train_test_split(tmp, test_size=0.5, random_state=42)

train_ds = LMDataset(train_texts, tokenizer)
val_ds   = LMDataset(val_texts, tokenizer)
test_ds  = LMDataset(test_texts, tokenizer)

coll = lambda b: collate_batch(b, tokenizer.pad_id)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, collate_fn=coll, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False, collate_fn=coll, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, collate_fn=coll, num_workers=2, pin_memory=True)

print("✅ Data ready | Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))


In [ ]:
class BiLSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim=512, num_layers=2,
                 dropout=0.3, embeddings=None, freeze_emb=False, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        if embeddings is not None:
            self.embedding.weight.data.copy_(torch.tensor(embeddings))
        self.embedding.weight.requires_grad = not freeze_emb
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            dropout=dropout, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim*2, vocab_size)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.fc(self.dropout(out))

model = BiLSTMLanguageModel(
    vocab_size=len(tokenizer.token_to_id),
    embed_dim=w2v_doc.vector_size,
    hidden_dim=512,
    num_layers=1,
    dropout=0.3,
    embeddings=tokenizer.embed_matrix,
    freeze_emb=False,
    pad_id=tokenizer.pad_id
).to(DEVICE)

print("✅ Model ready | Parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
@torch.no_grad()
def evaluate(model, loader, pad_id):
    model.eval()
    ce = nn.CrossEntropyLoss(ignore_index=pad_id)
    total_loss, total_tokens, total_correct = 0.0, 0, 0
    for inputs, targets, lengths in loader:
        inputs, targets, lengths = inputs.to(DEVICE), targets.to(DEVICE), lengths.to(DEVICE)
        logits = model(inputs, lengths)
        loss = ce(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        pred = torch.argmax(logits, dim=-1)
        mask = (targets != pad_id)
        total_correct += ((pred == targets) & mask).sum().item()
        total_loss += loss.item() * mask.sum().item()
        total_tokens += mask.sum().item()
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    acc = total_correct / max(1, total_tokens)
    return avg_loss, ppl, acc

def compute_bleu(model, loader, tokenizer, pad_id, max_batches=10):
    model.eval()
    smoothie = SmoothingFunction().method4
    scores = []
    for b_idx, (inputs, targets, lengths) in enumerate(loader):
        if b_idx >= max_batches: break
        inputs, targets, lengths = inputs.to(DEVICE), targets.to(DEVICE), lengths.to(DEVICE)
        logits = model(inputs, lengths)
        preds = torch.argmax(logits, dim=-1).cpu().tolist()
        tgts  = targets[:, :logits.size(1)].cpu().tolist()
        for hyp_ids, ref_ids in zip(preds, tgts):
            hyp = tokenizer.decode(hyp_ids).split()
            ref = [tokenizer.decode(ref_ids).split()]
            if len(hyp) and len(ref[0]):
                scores.append(sentence_bleu(ref, hyp, smoothing_function=smoothie))
    return float(np.mean(scores)) if scores else 0.0


In [ ]:
EPOCHS, patience, best_val, wait = 10, 4, float("inf"), 0
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

for ep in range(1, EPOCHS+1):
    model.train()
    total_loss, total_tokens, total_correct = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {ep}/{EPOCHS}", unit="batch", leave=False)

    for inputs, targets, lengths in pbar:
        inputs, targets, lengths = inputs.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True), lengths.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs, lengths)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        mask = (targets != tokenizer.pad_id)
        total_loss += loss.detach().cpu().item() * mask.sum().cpu().item()
        total_tokens += mask.sum().cpu().item()
        total_correct += ((torch.argmax(logits, dim=-1) == targets) & mask).sum().cpu().item()
        if total_tokens > 0:
            pbar.set_postfix({"TrainLoss": f"{(total_loss/total_tokens):.4f}"})

    train_loss = total_loss / max(1, total_tokens)
    train_acc = total_correct / max(1, total_tokens)

    if ep % 2 == 0 or ep == EPOCHS:
        val_loss, val_ppl, val_acc = evaluate(model, val_loader, tokenizer.pad_id)
        scheduler.step(val_loss)
        print(f"\nEpoch {ep}/{EPOCHS} | Train Loss={train_loss:.4f}, Train Acc={train_acc:.3f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.3f}, PPL={val_ppl:.2f}")
        if val_loss < best_val - 1e-5:
            best_val, wait = val_loss, 0
            torch.save(model.state_dict(), "bilstm_doc_best.pt")
        else:
            wait += 1
            if wait >= patience:
                print("⏹️ Early stopping.")
                break
    else:
        print(f"\nEpoch {ep}/{EPOCHS} | Train Loss={train_loss:.4f}, Train Acc={train_acc:.3f}")


In [ ]:
import os, torch
os.makedirs("/kaggle/working/models", exist_ok=True)
torch.save(model.state_dict(), "/kaggle/working/models/bilstm_doc_best.pt")
print("✅ Saved:", "/kaggle/working/models/bilstm_doc_best.pt")

# (optional) save a fuller checkpoint if you want to resume training later
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "best_val": '%.6f' % (locals().get('best_val', float('inf'))),
    "token_to_id": getattr(tokenizer, "token_to_id", None),
}, "/kaggle/working/models/bilstm_doc_full.ckpt")
print("✅ Saved:", "/kaggle/working/models/bilstm_doc_full.ckpt")


In [ ]:
if os.path.exists("bilstm_doc_best.pt"):
    print("🔄 Loading best model...")
    model.load_state_dict(torch.load("bilstm_doc_best.pt", map_location=DEVICE))
    model.eval()

    test_loss, test_ppl, test_acc = evaluate(model, test_loader, tokenizer.pad_id)
    bleu = compute_bleu(model, test_loader, tokenizer, tokenizer.pad_id)

    results = {
        "test_loss": round(test_loss, 4),
        "test_perplexity": round(test_ppl, 2),
        "test_accuracy": round(test_acc, 3),
        "bleu": round(bleu, 3)
    }
    with open("metrics_doc.json", "w") as f: json.dump(results, f, indent=2)
    print("✅ Final Results:", json.dumps(results, indent=2))
else:
    print("❌ No trained model found.")
